In [120]:
# Imports and API connection test
# Run once per session

import requests
import pandas as pd
import numpy as np
import torch
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import norm

# Your API key from the-odds-api.com
ODDS_API_KEY = 'ac3f07151eb58b517d62cf941c79755a'

# Test connection
url      = 'https://api.the-odds-api.com/v4/sports'
response = requests.get(url, params={'apiKey': ODDS_API_KEY})

print(f"Status: {response.status_code}")
print()

sports = response.json()
nba    = [s for s in sports if 'nba' in s['key'].lower()]

print("NBA markets available:")
for s in nba:
    print(f"  {s['key']:<30} {s['title']}")

Status: 200

NBA markets available:
  basketball_nba                 NBA
  basketball_nba_championship_winner NBA Championship Winner
  basketball_wnba                WNBA


In [122]:
# Load model and datasets
# Run once per session

import torch.nn as nn

# Load dataset
df_master = pd.read_csv('nba_master_dataset.csv', parse_dates=['GAME_DATE'])
df_master['HOME_AWAY'] = df_master['HOME_AWAY'].map({'HOME': 1, 'AWAY': 0})
df_master['POSITION']  = df_master['POSITION'].map({'G': 0, 'F': 1, 'C': 2})

# Add position-normalized rolling features
pos_roll_cols = [
    'OPP_PTS_VS_POS_roll5', 'OPP_REB_VS_POS_roll5',
    'OPP_AST_VS_POS_roll5', 'OPP_BLK_VS_POS_roll5',
    'OPP_STL_VS_POS_roll5', 'OPP_3PM_VS_POS_roll5'
]
for col in pos_roll_cols:
    norm_col = f'{col}_norm'
    df_master[norm_col] = df_master.groupby('POSITION')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

print(f"Dataset loaded: {len(df_master):,} rows")

# Load scaler
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print(f"Scaler loaded: {scaler.n_features_in_} features")

# Define features and targets
target_cols  = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']

feature_cols = [
    'PTS_roll5', 'REB_roll5', 'AST_roll5', 'BLK_roll5',
    'STL_roll5', 'FG3M_roll5', 'MIN_roll5', 'TOV_roll5',
    'FGA_roll5', 'FG3A_roll5',
    'PTS_roll10', 'REB_roll10', 'AST_roll10', 'BLK_roll10',
    'STL_roll10', 'FG3M_roll10', 'MIN_roll10', 'TOV_roll10',
    'FGA_roll10', 'FG3A_roll10',
    'HOME_AWAY', 'DAYS_REST', 'DEF_RATING', 'PACE',
    'OPP_PTS_ALLOWED_PG', 'OPP_REB_ALLOWED_PG', 'OPP_AST_ALLOWED_PG',
    'OPP_BLK_PG', 'OPP_STL_PG', 'OPP_3PM_ALLOWED_PG',
    'USG_PCT',
    'OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
    'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS',
    'USG_PCT_roll5',
    'OPP_PTS_VS_POS_roll5_norm', 'OPP_REB_VS_POS_roll5_norm',
    'OPP_AST_VS_POS_roll5_norm', 'OPP_BLK_VS_POS_roll5_norm',
    'OPP_STL_VS_POS_roll5_norm', 'OPP_3PM_VS_POS_roll5_norm',
    'IS_PLAYOFF',
    'RELATIVE_USG', 'USG_RANK',
    'PTS_std_roll10', 'REB_std_roll10',
    'PTS_cv_roll10', 'REB_cv_roll10',
]

print(f"Features: {len(feature_cols)}")

# Define model
class PlayerPropModel(nn.Module):
    def __init__(self, input_dim, target_stats):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.BatchNorm1d(128), nn.Dropout(0.5),
            nn.Linear(128, 64), nn.ReLU(),
            nn.BatchNorm1d(64), nn.Dropout(0.4),
        )
        self.heads = nn.ModuleDict({
            stat: nn.Sequential(
                nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2)
            ) for stat in target_stats
        })

    def forward(self, x):
        shared = self.trunk(x)
        outputs = {}
        for stat, head in self.heads.items():
            raw       = head(shared)
            mu        = raw[:, 0]
            log_sigma = torch.clamp(raw[:, 1], min=-3, max=3)
            sigma     = torch.exp(log_sigma) + 1e-6
            outputs[stat] = (mu, sigma)
        return outputs

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = PlayerPropModel(input_dim=len(feature_cols), target_stats=target_cols)
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

print(f"Model loaded ✅")
print(f"Device: {device}")

Dataset loaded: 56,985 rows
Scaler loaded: 51 features
Features: 51
Model loaded ✅
Device: cpu


In [124]:
# Prediction function
# Run once per session

def predict_player(player_name, df_master, model, scaler, feature_cols, target_cols):
    player_df = df_master[df_master['PLAYER_NAME'] == player_name].copy()

    if len(player_df) == 0:
        return None

    player_df = player_df.sort_values('GAME_DATE')
    latest    = player_df.iloc[-1]

    if player_df[feature_cols].iloc[-1].isna().any():
        complete_rows = player_df.dropna(subset=feature_cols)
        if len(complete_rows) == 0:
            return None
        latest = complete_rows.iloc[-1]

    features = latest[feature_cols].values.astype(np.float32).reshape(1, -1)
    features = scaler.transform(features)

    if np.isnan(features).any():
        return None

    feature_tensor = torch.tensor(features, dtype=torch.float32).to(device)

    with torch.no_grad():
        outputs = model(feature_tensor)

    results = {}
    for stat in target_cols:
        mu    = outputs[stat][0].item()
        sigma = outputs[stat][1].item()
        results[stat] = {'mu': mu, 'sigma': sigma}

    return results, latest['GAME_DATE'], len(player_df)

In [126]:
# Pull injury report for ESPN injury API
# Re-run freely to pull fresh injury reports

import requests

def get_injury_report():
    """
    Pulls NBA injury report from ESPN API.
    Returns dict of {player_name: status}
    """
    try:
        url = 'https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries'
        
        response = requests.get(url, timeout=10)
        print(f"Status: {response.status_code}")
        
        if response.status_code != 200:
            print(f"Error: {response.text[:200]}")
            return {}
        
        data     = response.json()
        injuries = {}
        
        # Parse injury data
        for team in data.get('injuries', []):
            team_name = team.get('team', {}).get('displayName', '')
            for injury in team.get('injuries', []):
                player_name = injury.get('athlete', {}).get('displayName', '')
                status      = injury.get('status', '')
                details     = injury.get('shortComment', '')
                
                injuries[player_name] = {
                    'status':  status,
                    'details': details,
                    'team':    team_name,
                }
        
        return injuries
    
    except Exception as e:
        print(f"Error: {e}")
        return {}

injury_report = get_injury_report()

print(f"Players on injury report: {len(injury_report)}")
print()
print("Sample:")
for player, info in list(injury_report.items())[:10]:
    print(f"  {player:<28} {info['status']:<15} {info['details']}")

Status: 200
Players on injury report: 126

Sample:
  Keshon Gilbert               Out             Gilbert signed a two-way contract with the Hawks on Tuesday, according to Jake Fischer of BleacherReport.com.
  Jock Landale                 Out             Landale (ankle) will be re-evaluated in two weeks, Brad Rowland of the Locked On Podcast Network reports.
  Jayson Tatum                 Out             Tatum (knee) has been ruled out for Saturday's Game 7 against the 76ers, Shams Charania of ESPN reports.
  Nolan Traore                 Out             Traore (illness) has been ruled out for Sunday's game against the Raptors.
  Nic Claxton                  Out             Claxton (finger) is out for Sunday's game in Toronto, Erik Slater of ClutchPoints.com reports.
  Ziaire Williams              Out             Williams (foot) will miss Sunday's regular-season finale against the Raptors, Erik Slater of ClutchPoints.com reports.
  Noah Clowney                 Out             Clowney (a

In [128]:
# Pull props from DraftKings
# Re-run freely to get fresh player props from each game (API calls limited to 500 per day)

from datetime import datetime, timezone, timedelta

# EST timezone
EST = timezone(timedelta(hours=-4))

# Pull tonight's NBA props from DraftKings
url      = 'https://api.the-odds-api.com/v4/sports/basketball_nba/odds'
response = requests.get(url, params={
    'apiKey':    ODDS_API_KEY,
    'regions':   'us',
    'oddsFormat': 'american',
})
games = response.json()

print(f"NBA games available: {len(games)}")
for g in games:
    utc_time = datetime.fromisoformat(g['commence_time'].replace('Z', '+00:00'))
    est_time = utc_time.astimezone(EST)
    print(f"  {g['away_team']} @ {g['home_team']}  —  {est_time.strftime('%I:%M %p EST  %Y-%m-%d')}")
print()

# Filter to only today's games in EST
today_est   = datetime.now(EST).strftime('%Y-%m-%d')
games_today = []

for g in games:
    utc_time = datetime.fromisoformat(g['commence_time'].replace('Z', '+00:00'))
    est_time = utc_time.astimezone(EST)
    if est_time.strftime('%Y-%m-%d') == today_est:
        games_today.append(g)

print(f"Games today in EST ({today_est}): {len(games_today)}")
for g in games_today:
    utc_time = datetime.fromisoformat(g['commence_time'].replace('Z', '+00:00'))
    est_time = utc_time.astimezone(EST)
    print(f"  {g['away_team']} @ {g['home_team']}  —  {est_time.strftime('%I:%M %p EST')}")
print()

# Use today's games for props pull
games = games_today

# Step 2 — Pull props for each game
MARKETS = [
    'player_points',
    'player_rebounds',
    'player_assists',
    'player_threes',
]

all_props = []

for game in games:
    game_id   = game['id']
    home_team = game['home_team']
    away_team = game['away_team']

    for market in MARKETS:
        url      = f'https://api.the-odds-api.com/v4/sports/basketball_nba/events/{game_id}/odds'
        response = requests.get(url, params={
            'apiKey':     ODDS_API_KEY,
            'regions':    'us',
            'markets':    market,
            'bookmakers': 'draftkings',
            'oddsFormat': 'american',
        })
        time.sleep(0.5)

        if response.status_code != 200:
            print(f"❌ Failed {market} for {away_team} @ {home_team}")
            continue

        data = response.json()

        for bookmaker in data.get('bookmakers', []):
            for mkt in bookmaker.get('markets', []):
                players_seen = {}
                for outcome in mkt.get('outcomes', []):
                    player = outcome['description']
                    side   = outcome['name']
                    price  = outcome['price']
                    line   = outcome['point']

                    if player not in players_seen:
                        players_seen[player] = {
                            'player':    player,
                            'stat':      market,
                            'line':      line,
                            'home_team': home_team,
                            'away_team': away_team,
                        }

                    if side == 'Over':
                        players_seen[player]['over_juice']  = price
                    else:
                        players_seen[player]['under_juice'] = price

                all_props.extend(players_seen.values())

# Map market names to model stat names
market_to_stat = {
    'player_points':   'PTS',
    'player_rebounds': 'REB',
    'player_assists':  'AST',
    'player_threes':   'FG3M',
}

df_props = pd.DataFrame(all_props)
df_props['stat'] = df_props['stat'].map(market_to_stat)

print(f"Total props pulled: {len(df_props)}")
print(f"Requests remaining: {response.headers.get('x-requests-remaining')}")
print()
print(df_props.head(10))

NBA games available: 1
  San Antonio Spurs @ New York Knicks  —  08:42 PM EST  2026-06-10

Games today in EST (2026-06-10): 1
  San Antonio Spurs @ New York Knicks  —  08:42 PM EST

Total props pulled: 69
Requests remaining: 267

               player stat  line        home_team          away_team  \
0       Jalen Brunson  PTS  27.5  New York Knicks  San Antonio Spurs   
1   Victor Wembanyama  PTS  26.5  New York Knicks  San Antonio Spurs   
2  Karl-Anthony Towns  PTS  17.5  New York Knicks  San Antonio Spurs   
3          OG Anunoby  PTS  16.5  New York Knicks  San Antonio Spurs   
4      Stephon Castle  PTS  16.5  New York Knicks  San Antonio Spurs   
5        De'Aaron Fox  PTS  14.5  New York Knicks  San Antonio Spurs   
6        Dylan Harper  PTS  14.5  New York Knicks  San Antonio Spurs   
7       Devin Vassell  PTS  12.5  New York Knicks  San Antonio Spurs   
8       Mikal Bridges  PTS  11.5  New York Knicks  San Antonio Spurs   
9           Josh Hart  PTS  10.5  New York Knicks 

In [129]:
# Injury-aware prediction pipeline
# Flags bets where the player or a key teammate is injured
# Re-run freely

def get_status_flag(player_name, injury_report):
    """
    Returns injury flag for a player.
    """
    if player_name in injury_report:
        status = injury_report[player_name]['status']
        if status.lower() == 'out':
            return '🚫'
        elif status.lower() in ['doubtful', 'questionable']:
            return '⚠️'
        else:
            return '📋'
    return ''


# Check which tonight's players are on the injury report
print("Injury report check for tonight's players:")
print()

tonight_players = df_props['player'].unique()
flagged = []

for player in sorted(tonight_players):
    if player in injury_report:
        info = injury_report[player]
        print(f"  {player:<28} {info['status']:<15} {info['details'][:60]}")
        flagged.append(player)

if len(flagged) == 0:
    print("  No players from tonight's props are on the injury report ✅")
print()
print(f"Total flagged: {len(flagged)}")

Injury report check for tonight's players:

  No players from tonight's props are on the injury report ✅

Total flagged: 0


In [132]:
# Run predictions and print table
# Re-run freely to get predictions on new props

# Stat-specific filters based on performance analysis
SKIP_STATS       = ['REB']   # skip entirely — edge not predictive
MIN_EDGE_FG3M    = 12.0      # raised threshold for FG3M
SUSPICIOUS_RATIO = 0.4       # remove OVER bets where line < 40% of model pred

# Updated NBA thresholds based on edge analysis
# Only 20%+ edge bets are historically profitable
MIN_EDGE_GREEN  = 20.0   # 100+ games
MIN_EDGE_YELLOW = 20.0   # 50-99 games
MIN_EDGE_RED    = 25.0   # <50 games

def american_to_prob(juice):
    if juice < 0:
        return abs(juice) / (abs(juice) + 100)
    else:
        return 100 / (juice + 100)

def confidence_tier(game_count):
    if game_count >= 100:
        return '🟢'
    elif game_count >= 50:
        return '🟡'
    else:
        return '🔴'

def get_injury_flag(player_name, injury_report):
    if player_name in injury_report:
        status = injury_report[player_name]['status'].lower()
        if status == 'out':
            return 'OUT'
        elif status == 'doubtful':
            return 'DOUBT'
        elif status == 'questionable':
            return 'QUEST'
        else:
            return 'INACT'
    return ''

# Run model on every prop
results_list = []

for _, row in df_props.iterrows():
    player = row['player']
    stat   = row['stat']
    line   = row['line']

    if 'over_juice' not in row or 'under_juice' not in row:
        continue
    if pd.isna(row['over_juice']) or pd.isna(row['under_juice']):
        continue

    pred = predict_player(
        player, df_master, model, scaler, feature_cols, target_cols
    )

    if pred is None:
        continue

    predictions, last_game, game_count = pred

    if stat not in predictions:
        continue

    mu    = predictions[stat]['mu']
    sigma = predictions[stat]['sigma']

    prob_over  = 1 - norm.cdf(line, mu, sigma)
    prob_under = norm.cdf(line, mu, sigma)

    breakeven_over  = american_to_prob(row['over_juice'])
    breakeven_under = american_to_prob(row['under_juice'])

    edge_over  = prob_over  - breakeven_over
    edge_under = prob_under - breakeven_under

    if edge_over > edge_under and edge_over > 0:
        recommendation = 'OVER'
        edge = edge_over
    elif edge_under > edge_over and edge_under > 0:
        recommendation = 'UNDER'
        edge = edge_under
    else:
        recommendation = 'NO BET'
        edge = max(edge_over, edge_under)

    injury_flag = get_injury_flag(player, injury_report)

    results_list.append({
        'player':         player,
        'stat':           stat,
        'line':           line,
        'mu':             round(mu, 1),
        'sigma':          round(sigma, 1),
        'recommendation': recommendation,
        'edge':           round(edge * 100, 1),
        'over_juice':     row['over_juice'],
        'under_juice':    row['under_juice'],
        'game':           f"{row['away_team']} @ {row['home_team']}",
        'data_through':   str(last_game.date()),
        'game_count':     game_count,
        'confidence':     confidence_tier(game_count),
        'injury':         injury_flag,
    })

df_results = pd.DataFrame(results_list)

# Separate injured players
df_injured = df_results[df_results['injury'] == 'OUT']

# Filter bets with stat-specific rules and updated thresholds
df_bets = df_results[
    (df_results['recommendation'] != 'NO BET') &
    (df_results['injury'] != 'OUT') &
    (~df_results['stat'].isin(SKIP_STATS)) &
    (
        # FG3M — stricter threshold
        (
            (df_results['stat'] == 'FG3M') &
            (df_results['edge'] >= MIN_EDGE_FG3M)
        ) |
        # PTS and AST — tiered thresholds by confidence
        (
            (df_results['stat'].isin(['PTS', 'AST'])) &
            (
                ((df_results['game_count'] >= 100) &
                 (df_results['edge'] >= MIN_EDGE_GREEN)) |
                ((df_results['game_count'] >= 50) &
                 (df_results['game_count'] < 100) &
                 (df_results['edge'] >= MIN_EDGE_YELLOW)) |
                ((df_results['game_count'] < 50) &
                 (df_results['edge'] >= MIN_EDGE_RED))
            )
        )
    ) &
    # Remove suspicious lines where book prices far below model pred
    ~(
        (df_results['recommendation'] == 'OVER') &
        (df_results['mu'] > 0) &
        (df_results['line'] / df_results['mu'] < SUSPICIOUS_RATIO)
    )
].sort_values('edge', ascending=False).reset_index(drop=True)

# Print ranked table
print(f"{'='*100}")
print(f"  TONIGHT'S BEST BETS — DraftKings Props")
print(f"{'='*100}")
print(f"{'#':<4} {'C':<3} {'Player':<28} {'Stat':<6} {'Line':<7} "
      f"{'Pred μ':<8} {'Rec':<7} {'Edge':>6}  {'Juice':<8} {'Games'}")
print(f"{'-'*100}")

for i, row in df_bets.iterrows():
    juice     = row['over_juice'] if row['recommendation'] == 'OVER' else row['under_juice']
    juice_str = f"+{juice}" if juice > 0 else str(juice)
    inj       = f" ⚠️ {row['injury']}" if row['injury'] else ''
    print(
        f"{i+1:<4} {row['confidence']:<3} {row['player']:<28} {row['stat']:<6} "
        f"{row['line']:<7} {row['mu']:<8} {row['recommendation']:<7} "
        f"{row['edge']:>5.1f}%  {juice_str:<8} {row['game_count']}"
        f"{inj}"
    )

print(f"{'='*100}")
print()

if len(df_injured) > 0:
    print(f"🚫 REMOVED — Player listed as OUT:")
    for _, row in df_injured.iterrows():
        if row['recommendation'] != 'NO BET':
            print(f"   {row['player']:<28} {row['stat']:<6} "
                  f"Line {row['line']}  Edge {row['edge']:.1f}%  ← removed")
    print()

pts_bets  = len(df_bets[df_bets['stat'] == 'PTS'])
ast_bets  = len(df_bets[df_bets['stat'] == 'AST'])
fg3m_bets = len(df_bets[df_bets['stat'] == 'FG3M'])

high   = len(df_bets[df_bets['game_count'] >= 100])
medium = len(df_bets[(df_bets['game_count'] >= 50) & (df_bets['game_count'] < 100)])
low    = len(df_bets[df_bets['game_count'] < 50])

print(f"🟢 High confidence (100+ games):    {high} bets  (min edge {MIN_EDGE_GREEN}%)")
print(f"🟡 Medium confidence (50-99 games): {medium} bets  (min edge {MIN_EDGE_YELLOW}%)")
print(f"🔴 Low confidence (<50 games):      {low} bets  (min edge {MIN_EDGE_RED}%)")
print()
print(f"By stat:  PTS {pts_bets}  AST {ast_bets}  "
      f"FG3M {fg3m_bets}  REB skipped")
print()
print(f"Total +EV bets shown: {len(df_bets)}")
print(f"Total props scanned:  {len(df_results)}")
print(f"Removed (OUT):        {len(df_injured[df_injured['recommendation'] != 'NO BET'])}")
print()
print(f"⚙️  Active filters: REB skipped | FG3M min {MIN_EDGE_FG3M}% | "
      f"Suspicious lines removed (ratio < {SUSPICIOUS_RATIO}) | "
      f"Min edge {MIN_EDGE_GREEN}%")

  TONIGHT'S BEST BETS — DraftKings Props
#    C   Player                       Stat   Line    Pred μ   Rec       Edge  Juice    Games
----------------------------------------------------------------------------------------------------
1    🟢   Keldon Johnson               PTS    6.5     12.7     OVER     30.0%  -119     322
2    🟢   Jalen Brunson                PTS    27.5    21.1     UNDER    26.6%  -116     367
3    🟢   Victor Wembanyama            PTS    26.5    21.7     UNDER    23.5%  -104     107
4    🟢   Mikal Bridges                AST    2.5     4.8      OVER     22.3%  -147     408
5    🟢   Mikal Bridges                PTS    11.5    16.7     OVER     21.6%  -125     408
6    🟢   OG Anunoby                   AST    1.5     2.8      OVER     21.1%  -128     272
7    🟢   De'Aaron Fox                 FG3M   1.5     2.1      OVER     20.0%  +125     301

🟢 High confidence (100+ games):    7 bets  (min edge 20.0%)
🟡 Medium confidence (50-99 games): 0 bets  (min edge 20.0%)
🔴 Low c

In [134]:
# CLV Tracking
# Records every bet recommendation and tracks actual outcomes
# This is the only metric that matters for measuring real edge
# Only run once a day as it is used to log bets and write to file

import json
import os
from datetime import datetime

BETS_LOG_FILE = 'bets_log.json'

def load_bets_log():
    if os.path.exists(BETS_LOG_FILE):
        with open(BETS_LOG_FILE, 'r') as f:
            return json.load(f)
    return []

def save_bets_log(log):
    with open(BETS_LOG_FILE, 'w') as f:
        json.dump(log, f, indent=2)

def log_todays_bets(df_bets, injury_report, min_edge=5.0):
    """Logs today's recommended bets. Call once before games start."""
    log        = load_bets_log()
    today      = datetime.now().strftime('%Y-%m-%d')
    bets_added = 0

    for _, row in df_bets[df_bets['edge'] >= min_edge].iterrows():
        juice = row['over_juice'] if row['recommendation'] == 'OVER' else row['under_juice']

        bet_entry = {
            'date':           today,
            'player':         row['player'],
            'stat':           row['stat'],
            'line':           row['line'],
            'recommendation': row['recommendation'],
            'edge':           row['edge'],
            'juice':          juice,
            'pred_mu':        row['mu'],
            'pred_sigma':     row['sigma'],
            'game':           row['game'],
            'game_count':     row['game_count'],
            'confidence':     row['confidence'],
            'data_through':   row['data_through'],
            'closing_line':   None,
            'actual_result':  None,
            'bet_result':     None,
            'clv':            None,
        }

        is_duplicate = any(
            b['date']   == today and
            b['player'] == row['player'] and
            b['stat']   == row['stat']
            for b in log
        )

        if not is_duplicate:
            log.append(bet_entry)
            bets_added += 1

    save_bets_log(log)
    print(f"✅ Logged {bets_added} bets for {today}")
    return log


def update_bet_result(date, player, stat, actual_result, closing_line=None):
    """Updates a bet with actual result after the game."""
    log     = load_bets_log()
    updated = False

    for bet in log:
        if (bet['date']   == date and
            bet['player'] == player and
            bet['stat']   == stat):

            bet['actual_result'] = actual_result
            bet['closing_line']  = closing_line

            if bet['recommendation'] == 'OVER':
                if actual_result > bet['line']:
                    bet['bet_result'] = 'WIN'
                elif actual_result == bet['line']:
                    bet['bet_result'] = 'PUSH'
                else:
                    bet['bet_result'] = 'LOSS'
            else:
                if actual_result < bet['line']:
                    bet['bet_result'] = 'WIN'
                elif actual_result == bet['line']:
                    bet['bet_result'] = 'PUSH'
                else:
                    bet['bet_result'] = 'LOSS'

            if closing_line is not None:
                if bet['recommendation'] == 'OVER':
                    bet['clv'] = round(bet['line'] - closing_line, 1)
                else:
                    bet['clv'] = round(closing_line - bet['line'], 1)

            updated = True
            break

    if updated:
        save_bets_log(log)
        print(f"✅ Updated {player} {stat} — "
              f"Result: {actual_result}  "
              f"Bet: {bet['recommendation']} {bet['line']}  "
              f"Outcome: {bet['bet_result']}")
    else:
        print(f"❌ Bet not found: {player} {stat} on {date}")


def print_performance_summary():
    """Prints full performance summary from bets log."""
    log = load_bets_log()

    if len(log) == 0:
        print("No bets logged yet.")
        return

    completed = [b for b in log if b['bet_result'] is not None]
    pending   = [b for b in log if b['bet_result'] is None]

    print(f"{'='*60}")
    print(f"  PERFORMANCE SUMMARY")
    print(f"{'='*60}")
    print(f"  Total bets logged:   {len(log)}")
    print(f"  Completed:           {len(completed)}")
    print(f"  Pending:             {len(pending)}")
    print()

    if len(completed) == 0:
        print("  No completed bets yet.")
        return

    wins     = len([b for b in completed if b['bet_result'] == 'WIN'])
    losses   = len([b for b in completed if b['bet_result'] == 'LOSS'])
    pushes   = len([b for b in completed if b['bet_result'] == 'PUSH'])
    win_rate = wins / (wins + losses) if (wins + losses) > 0 else 0

    print(f"  Record:              {wins}W - {losses}L - {pushes}P")
    print(f"  Win rate:            {win_rate:.1%}")
    print()

    clv_bets = [b for b in completed if b['clv'] is not None]
    if len(clv_bets) > 0:
        avg_clv = sum(b['clv'] for b in clv_bets) / len(clv_bets)
        pos_clv = len([b for b in clv_bets if b['clv'] > 0])
        print(f"  Avg CLV:             {avg_clv:+.2f} points")
        print(f"  Positive CLV bets:   {pos_clv}/{len(clv_bets)}")
        print()

    print(f"  By stat:")
    for stat in ['PTS', 'REB', 'AST', 'FG3M']:
        stat_bets  = [b for b in completed if b['stat'] == stat]
        if len(stat_bets) == 0:
            continue
        stat_wins  = len([b for b in stat_bets if b['bet_result'] == 'WIN'])
        stat_loss  = len(stat_bets) - stat_wins
        stat_rate  = stat_wins / len(stat_bets)
        print(f"    {stat:<6} {stat_wins}W/{stat_loss}L  ({stat_rate:.0%})")

    print()
    print(f"  By confidence:")
    for tier in ['🟢', '🟡', '🔴']:
        tier_bets  = [b for b in completed if b['confidence'] == tier]
        if len(tier_bets) == 0:
            continue
        tier_wins  = len([b for b in tier_bets if b['bet_result'] == 'WIN'])
        tier_loss  = len(tier_bets) - tier_wins
        tier_rate  = tier_wins / len(tier_bets)
        print(f"    {tier}  {tier_wins}W/{tier_loss}L  ({tier_rate:.0%})")

    print(f"{'='*60}")
    print()
    print(f"  Recent bets:")
    print(f"  {'Date':<12} {'Player':<25} {'Stat':<6} {'Line':<7} "
          f"{'Rec':<7} {'Actual':<8} {'Result':<7} {'CLV'}")
    print(f"  {'-'*85}")

    for bet in sorted(completed, key=lambda x: x['date'], reverse=True)[:20]:
        clv_str = f"{bet['clv']:+.1f}" if bet['clv'] is not None else 'N/A'
        print(
            f"  {bet['date']:<12} {bet['player']:<25} {bet['stat']:<6} "
            f"{bet['line']:<7} {bet['recommendation']:<7} "
            f"{str(bet['actual_result']):<8} {bet['bet_result']:<7} {clv_str}"
        )


# ── Run it ─────────────────────────────────────────
# Log today's bets
log = log_todays_bets(df_bets, injury_report, min_edge=5.0)
print()

# Show what was logged
today       = datetime.now().strftime('%Y-%m-%d')
todays_bets = [b for b in log if b['date'] == today]

print("Bets logged today:")
print(f"{'Player':<28} {'Stat':<6} {'Line':<7} {'Rec':<7} {'Edge':>6}")
print("-" * 60)
for bet in todays_bets:
    print(f"{bet['player']:<28} {bet['stat']:<6} {bet['line']:<7} "
          f"{bet['recommendation']:<7} {bet['edge']:>5.1f}%")

✅ Logged 7 bets for 2026-06-10

Bets logged today:
Player                       Stat   Line    Rec       Edge
------------------------------------------------------------
Keldon Johnson               PTS    6.5     OVER     30.0%
Jalen Brunson                PTS    27.5    UNDER    26.6%
Victor Wembanyama            PTS    26.5    UNDER    23.5%
Mikal Bridges                AST    2.5     OVER     22.3%
Mikal Bridges                PTS    11.5    OVER     21.6%
OG Anunoby                   AST    1.5     OVER     21.1%
De'Aaron Fox                 FG3M   1.5     OVER     20.0%


In [136]:
# Automated results updater
# Run once per session

from nba_api.stats.endpoints import playergamelog
import time
import numpy as np

def save_closing_lines(df_bets, min_edge=5.0, game_filter=None):
    """
    Saves closing lines for logged bets.
    Use game_filter to save lines for specific tipoff windows.

    Example:
      save_closing_lines(df_bets, game_filter='Oklahoma City Thunder')
      save_closing_lines(df_bets, game_filter='Los Angeles Lakers')
      save_closing_lines(df_bets)  # saves all games at once
    """
    log   = load_bets_log()
    today = datetime.now().strftime('%Y-%m-%d')

    url      = 'https://api.the-odds-api.com/v4/sports/basketball_nba/odds'
    response = requests.get(url, params={
        'apiKey':    ODDS_API_KEY,
        'regions':   'us',
        'oddsFormat': 'american',
    })
    games = response.json()

    if game_filter:
        games = [
            g for g in games
            if game_filter in g['home_team'] or game_filter in g['away_team']
        ]
        print(f"Filtering to games involving: {game_filter}")
        print(f"Games found: {len(games)}")

    current_lines = {}

    for game in games:
        game_id = game['id']

        for market in ['player_points', 'player_rebounds',
                       'player_assists', 'player_threes']:

            url      = f'https://api.the-odds-api.com/v4/sports/basketball_nba/events/{game_id}/odds'
            response = requests.get(url, params={
                'apiKey':     ODDS_API_KEY,
                'regions':    'us',
                'markets':    market,
                'bookmakers': 'draftkings',
                'oddsFormat': 'american',
            })
            time.sleep(0.5)

            if response.status_code != 200:
                continue

            data = response.json()

            market_to_stat = {
                'player_points':   'PTS',
                'player_rebounds': 'REB',
                'player_assists':  'AST',
                'player_threes':   'FG3M',
            }
            stat = market_to_stat[market]

            for bookmaker in data.get('bookmakers', []):
                for mkt in bookmaker.get('markets', []):
                    players_seen = {}
                    for outcome in mkt.get('outcomes', []):
                        player = outcome['description']
                        if player not in players_seen:
                            players_seen[player] = outcome['point']
                    for player, line in players_seen.items():
                        current_lines[(player, stat)] = line

    updated = 0
    for bet in log:
        if bet['date'] == today and bet['closing_line'] is None:
            key = (bet['player'], bet['stat'])
            if key in current_lines:
                bet['closing_line'] = current_lines[key]
                updated += 1

    save_bets_log(log)
    print(f"✅ Closing lines saved for {updated} bets")
    return current_lines


def auto_update_results(date=None):
    """
    Automatically pulls actual stats from NBA API and updates bet results.
    Run 30-60 min after final buzzer.

    date: optional override — defaults to today
          use '2026-05-29' to update a previous day's bets
    """
    log         = load_bets_log()
    target_date = date if date else datetime.now().strftime('%Y-%m-%d')

    pending = [
        b for b in log
        if b['date'] == target_date and b['bet_result'] is None
    ]

    if len(pending) == 0:
        print(f"No pending bets to update for {target_date}.")
        return

    print(f"Updating {len(pending)} pending bets for {target_date}...")
    print()

    players_needed = list(set(b['player'] for b in pending))
    player_stats   = {}

    for player_name in players_needed:
        player_rows = df_master[df_master['PLAYER_NAME'] == player_name]

        if len(player_rows) == 0:
            print(f"  ⚠️  {player_name} not found in dataset")
            continue

        player_id = player_rows['Player_ID'].iloc[0]

        try:
            # Try playoffs first — NBA Finals are currently ongoing
            log_data = playergamelog.PlayerGameLog(
                player_id=player_id,
                season='2025-26',
                season_type_all_star='Playoffs'
            )
            time.sleep(1.5)
            df_log = log_data.get_data_frames()[0]

            # Fall back to regular season if no playoff games
            if len(df_log) == 0:
                log_data = playergamelog.PlayerGameLog(
                    player_id=player_id,
                    season='2025-26',
                )
                time.sleep(1.5)
                df_log = log_data.get_data_frames()[0]

            if len(df_log) == 0:
                print(f"  ⚠️  No games found for {player_name}")
                continue

            df_log['GAME_DATE'] = pd.to_datetime(df_log['GAME_DATE'], format='mixed')
            latest    = df_log.sort_values('GAME_DATE').iloc[-1]
            game_date = latest['GAME_DATE'].strftime('%Y-%m-%d')

            player_stats[player_name] = {
                'PTS':  int(latest['PTS']),
                'REB':  int(latest['REB']),
                'AST':  int(latest['AST']),
                'FG3M': int(latest['FG3M']),
                'date': game_date,
            }

            print(f"  ✅ {player_name:<28} "
                  f"PTS:{latest['PTS']:.0f}  "
                  f"REB:{latest['REB']:.0f}  "
                  f"AST:{latest['AST']:.0f}  "
                  f"FG3M:{latest['FG3M']:.0f}  "
                  f"({game_date})")

        except Exception as e:
            print(f"  ❌ {player_name} — {e}")
            continue

    print()

    updated = 0
    skipped = 0

    for bet in log:
        if bet['date'] != target_date or bet['bet_result'] is not None:
            continue

        player = bet['player']
        stat   = bet['stat']

        if player not in player_stats:
            skipped += 1
            continue

        stats     = player_stats[player]
        game_date = stats['date']

        if game_date != target_date:
            print(f"  ⚠️  {player} {stat} — most recent game was {game_date}, not {target_date}")
            skipped += 1
            continue

        actual = stats[stat]

        if bet['recommendation'] == 'OVER':
            if actual > bet['line']:
                result = 'WIN'
            elif actual == bet['line']:
                result = 'PUSH'
            else:
                result = 'LOSS'
        else:
            if actual < bet['line']:
                result = 'WIN'
            elif actual == bet['line']:
                result = 'PUSH'
            else:
                result = 'LOSS'

        bet['actual_result'] = actual
        bet['bet_result']    = result

        if bet['closing_line'] is not None:
            if bet['recommendation'] == 'OVER':
                bet['clv'] = round(bet['line'] - bet['closing_line'], 1)
            else:
                bet['clv'] = round(bet['closing_line'] - bet['line'], 1)

        icon = '✅' if result == 'WIN' else '❌' if result == 'LOSS' else '➡️'
        print(f"  {icon} {player:<28} {stat:<6} "
              f"{bet['recommendation']} {bet['line']}  "
              f"Actual: {actual}  {result}")
        updated += 1

    save_bets_log(log)
    print()
    print(f"Updated: {updated}  Skipped: {skipped}")
    print()
    print_performance_summary()


print("✅ Automated functions loaded")
print()

✅ Automated functions loaded



In [138]:
# Run this 5 minutes before each tipoff window
# Change game_filter to match teams in that window

# 8:00 PM EST game 5/30/2026:
save_closing_lines(df_bets, game_filter='New York Knicks')

# 10:00 PM EST games (run separately before those tip off):
# save_closing_lines(df_bets, game_filter='Los Angeles Lakers')

Filtering to games involving: New York Knicks
Games found: 1
✅ Closing lines saved for 7 bets


{('Jalen Brunson', 'PTS'): 27.5,
 ('Victor Wembanyama', 'PTS'): 26.5,
 ('Karl-Anthony Towns', 'PTS'): 17.5,
 ('OG Anunoby', 'PTS'): 16.5,
 ('Stephon Castle', 'PTS'): 16.5,
 ("De'Aaron Fox", 'PTS'): 14.5,
 ('Dylan Harper', 'PTS'): 14.5,
 ('Devin Vassell', 'PTS'): 12.5,
 ('Mikal Bridges', 'PTS'): 11.5,
 ('Josh Hart', 'PTS'): 10.5,
 ('Julian Champagnie', 'PTS'): 9.5,
 ('Landry Shamet', 'PTS'): 7.5,
 ('Keldon Johnson', 'PTS'): 6.5,
 ('Miles McBride', 'PTS'): 4.5,
 ('Mitchell Robinson', 'PTS'): 3.5,
 ('Jordan Clarkson', 'PTS'): 2.5,
 ('Jose Alvarado', 'PTS'): 2.5,
 ('Luke Kornet', 'PTS'): 1.5,
 ('Carter Bryant', 'PTS'): 0.5,
 ('Victor Wembanyama', 'REB'): 11.5,
 ('Karl-Anthony Towns', 'REB'): 11.5,
 ('Josh Hart', 'REB'): 8.5,
 ('Dylan Harper', 'REB'): 6.5,
 ('OG Anunoby', 'REB'): 5.5,
 ('Julian Champagnie', 'REB'): 4.5,
 ('Stephon Castle', 'REB'): 4.5,
 ('Devin Vassell', 'REB'): 4.5,
 ('Mitchell Robinson', 'REB'): 4.5,
 ('Mikal Bridges', 'REB'): 3.5,
 ("De'Aaron Fox", 'REB'): 3.5,
 ('Jalen 

In [ ]:
# Safety reload — runs fast if already loaded, reloads if kernel died
import os

if 'df_master' not in dir() or 'model' not in dir():
    print("⚠️  Kernel was restarted — reloading...")
    %run "NBA_Live_Pipeline.ipynb"  # re-runs the whole notebook
else:
    print("✅ Everything already loaded")

auto_update_results()

In [116]:
# Run this 30-60 min after the final game ends
auto_update_results(date='2026-06-08')

Updating 10 pending bets for 2026-06-08...

  ✅ Josh Hart                    PTS:16  REB:9  AST:5  FG3M:4  (2026-06-08)
  ✅ Jalen Brunson                PTS:32  REB:5  AST:5  FG3M:3  (2026-06-08)
  ✅ Mikal Bridges                PTS:2  REB:5  AST:2  FG3M:0  (2026-06-08)
  ✅ Victor Wembanyama            PTS:32  REB:8  AST:6  FG3M:2  (2026-06-08)
  ✅ Keldon Johnson               PTS:7  REB:2  AST:0  FG3M:0  (2026-06-08)
  ✅ De'Aaron Fox                 PTS:12  REB:3  AST:8  FG3M:0  (2026-06-08)
  ✅ OG Anunoby                   PTS:28  REB:5  AST:1  FG3M:3  (2026-06-08)
  ✅ Miles McBride                PTS:0  REB:0  AST:0  FG3M:0  (2026-06-08)

  ❌ Miles McBride                AST    OVER 1.5  Actual: 0  LOSS
  ✅ Keldon Johnson               PTS    OVER 5.5  Actual: 7  WIN
  ❌ Miles McBride                PTS    OVER 5.5  Actual: 0  LOSS
  ❌ Miles McBride                FG3M   OVER 1.5  Actual: 0  LOSS
  ❌ Victor Wembanyama            PTS    UNDER 27.5  Actual: 32  LOSS
  ❌ Jalen Brunson 

In [118]:
# NBA Edge accuracy analysis
import numpy as np

# Load NBA bets log
nba_log       = load_bets_log()  # loads bets_log.json
nba_completed = [b for b in nba_log if b['bet_result'] is not None
                 and b['bet_result'] != 'VOID']

print(f"Total completed NBA bets: {len(nba_completed)}")
print()

# Bucket bets by edge range
buckets = {
    '5-10%':   [],
    '10-15%':  [],
    '15-20%':  [],
    '20-25%':  [],
    '25%+':    [],
}

for bet in nba_completed:
    edge = bet['edge']
    if edge >= 25:
        buckets['25%+'].append(bet)
    elif edge >= 20:
        buckets['20-25%'].append(bet)
    elif edge >= 15:
        buckets['15-20%'].append(bet)
    elif edge >= 10:
        buckets['10-15%'].append(bet)
    elif edge >= 5:
        buckets['5-10%'].append(bet)

print(f"{'Edge Range':<12} {'Bets':>6} {'Wins':>6} {'Losses':>6} "
      f"{'Win %':>8} {'vs Breakeven':>14}")
print("-" * 55)

for bucket, bets in buckets.items():
    if len(bets) == 0:
        continue
    wins   = len([b for b in bets if b['bet_result'] == 'WIN'])
    losses = len([b for b in bets if b['bet_result'] == 'LOSS'])
    total  = wins + losses
    if total == 0:
        continue
    win_rate   = wins / total
    breakeven  = 0.524
    vs_be      = win_rate - breakeven
    bar        = '█' * int(win_rate * 20)
    print(f"  {bucket:<10} {total:>6} {wins:>6} {losses:>6} "
          f"{win_rate:>8.1%} {vs_be:>+13.1%}  {bar}")

print()

# Break down by stat and edge
print(f"{'Stat':<8} {'Edge 5-15%':>12} {'Edge 15%+':>12}")
print("-" * 35)

for stat in ['PTS', 'REB', 'AST', 'FG3M']:
    low_bets  = [b for b in nba_completed
                 if b['stat'] == stat and 5 <= b['edge'] < 15]
    high_bets = [b for b in nba_completed
                 if b['stat'] == stat and b['edge'] >= 15]

    def win_rate_str(bets):
        if len(bets) == 0:
            return 'N/A'
        wins = len([b for b in bets if b['bet_result'] == 'WIN'])
        return f"{wins/len(bets):.0%} ({len(bets)})"

    print(f"  {stat:<6}  {win_rate_str(low_bets):>12}  "
          f"{win_rate_str(high_bets):>12}")

print()

# Correlation between edge and outcome
edges    = [b['edge'] for b in nba_completed]
outcomes = [1 if b['bet_result'] == 'WIN' else 0
            for b in nba_completed]
corr     = np.corrcoef(edges, outcomes)[0, 1]

print(f"Correlation between edge and outcome: {corr:+.3f}")
print()
if corr > 0.05:
    print("✅ Positive correlation — higher edge bets winning more often")
elif corr > -0.05:
    print("➡️  Near zero correlation — edge not yet predictive")
else:
    print("⚠️  Negative correlation — higher edge bets winning less often")

print()

# Compare NBA vs WNBA summary
print(f"{'='*40}")
print(f"  NBA vs WNBA Edge Comparison")
print(f"{'='*40}")
print(f"  NBA completed bets:  {len(nba_completed)}")
print()
nba_wins     = len([b for b in nba_completed if b['bet_result'] == 'WIN'])
nba_losses   = len([b for b in nba_completed if b['bet_result'] == 'LOSS'])
nba_win_rate = nba_wins / (nba_wins + nba_losses) if (nba_wins + nba_losses) > 0 else 0
print(f"  NBA record:          {nba_wins}W - {nba_losses}L ({nba_win_rate:.1%})")
print()
print(f"  By stat:")
for stat in ['PTS', 'REB', 'AST', 'FG3M']:
    stat_bets = [b for b in nba_completed if b['stat'] == stat]
    if len(stat_bets) == 0:
        continue
    stat_wins = len([b for b in stat_bets if b['bet_result'] == 'WIN'])
    print(f"    {stat:<6} {stat_wins}W/{len(stat_bets)-stat_wins}L  "
          f"({stat_wins/len(stat_bets):.0%})")

Total completed NBA bets: 87

Edge Range     Bets   Wins Losses    Win %   vs Breakeven
-------------------------------------------------------
  5-10%          17      5     12    29.4%        -23.0%  █████
  10-15%         11      4      7    36.4%        -16.0%  ███████
  15-20%         12      6      6    50.0%         -2.4%  ██████████
  20-25%         22     13      9    59.1%         +6.7%  ███████████
  25%+           25     12     13    48.0%         -4.4%  █████████

Stat       Edge 5-15%    Edge 15%+
-----------------------------------
  PTS          33% (9)      57% (23)
  REB          33% (6)       43% (7)
  AST          40% (5)      56% (16)
  FG3M         25% (8)      46% (13)

Correlation between edge and outcome: +0.170

✅ Positive correlation — higher edge bets winning more often

  NBA vs WNBA Edge Comparison
  NBA completed bets:  87

  NBA record:          40W - 47L (46.0%)

  By stat:
    PTS    16W/16L  (50%)
    REB    5W/8L  (38%)
    AST    11W/10L  (52%)
    